<a href="https://colab.research.google.com/github/kapileswaraputejasri/25B21CS063_DriveReady_opc_2026/blob/main/agenticAI_day2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install -q openai

In [3]:
# once per Colab session

from google.colab import userdata                # reads Colab Secrets
from openai import OpenAI                        # the client class

client = OpenAI(
    api_key=userdata.get("groq_api"),        # who you are, who pays
    base_url="https://api.groq.com/openai/v1",   # send requests to Groq, not OpenAI
)

response = client.chat.completions.create(
    model="openai/gpt-oss-120b",                  # which model answers
    messages=[
        {"role": "system", "content": "You are a funny joker"},
        {"role": "user",   "content": "tell me a joke"},
    ],
)
print(response.choices[0].message.content)

Why don’t scientists trust atoms anymore?

Because they **make up** everything! 😄


In [4]:
SYSTEM = """ROLE:   You are the assistant for Sweet Crumbs Home Bakery.
FACTS:  We bake to order. Orders need 24 hours' notice. All cakes have eggless versions.
        Plum cake Rs 500. Chocolate cake Rs 650. Cupcake Rs 80.
RULES:  Never promise a delivery time. If the answer isn't in FACTS, say "I'll check with the owner."
FORMAT: Under 50 words, friendly, English or Telugu matching the customer."""

response = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    temperature=0.2,                              # facts and prices: keep it focused
    messages=[{"role": "system", "content": SYSTEM},
              {"role": "user",   "content": "Can I get an eggless cake today evening?"}],
)
print(response.choices[0].message.content)
print(response.usage)

We need at least 24 hours’ notice, so we can’t prepare an eggless cake for this evening. We’d be happy to schedule it for tomorrow or later—just let us know!
CompletionUsage(completion_tokens=201, prompt_tokens=183, total_tokens=384, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=None, audio_tokens=None, reasoning_tokens=151, rejected_prediction_tokens=None), prompt_tokens_details=None, queue_time=0.789793011, prompt_time=0.008370415, completion_time=0.429511117, total_time=0.437881532)


In [11]:
MENU = {"plum cake": 500, "chocolate cake": 650, "cupcake": 80}   # key → value

In [16]:
MENU.get("donut", "donuts not available")

'donuts not available'

In [17]:
order = {"name": "Ravi", "item": "plum cake", "qty": 2, "eggless": True}

In [18]:
order["qty"] * MENU[order["item"]] # 1000

1000

In [19]:
import json
text = json.dumps(order)           # dict → JSON string, to send or save
back = json.loads(text)

In [20]:
text

'{"name": "Ravi", "item": "plum cake", "qty": 2, "eggless": true}'

In [21]:
back

{'name': 'Ravi', 'item': 'plum cake', 'qty': 2, 'eggless': True}

In [22]:
def get_price(item: str, qty: int = 1) -> int:
    return MENU[item.lower().strip()] * qty

get_price("cupcake", 6)    # 480
get_price("Plum Cake")

500

In [23]:
def get_price(item, qty=1) :
    """Return the total price in rupees for qty of an item on the Sweet Crumbs menu."""
    return MENU[item.lower().strip()] * qty

get_price("cupcake", 6)    # 480
get_price("Plum Cake")

500

In [24]:
a=90
print(a+b)
print('hi')

NameError: name 'b' is not defined

In [25]:
try:
  a=10
  print(a+b)
  print('hi')
except:
  print('b is not defined')

b is not defined


In [26]:
import json, functools
from openai import RateLimitError, APIConnectionError

MODEL = "openai/gpt-oss-120b"
MENU = {"plum cake": 500, "chocolate cake": 650, "cupcake": 80}

SYSTEM = f"""ROLE:   You are the friendly assistant for Sweet Crumbs Home Bakery.
FACTS:  Menu with prices in rupees: {json.dumps(MENU)}
        Orders need 24 hours' notice. All cakes have eggless versions.
RULES:  Never promise a delivery time. If it isn't in FACTS, say "I'll check with the owner."
        Remember the customer's name and order during the chat.
FORMAT: Under 50 words, friendly, English or Telugu matching the customer."""

def log_call(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        result = func(*args, **kwargs)
        print(f"   [log] {func.__name__}{args} -> {result!r}")
        return result
    return wrapper

@log_call
def get_price(item: str, qty: int = 1) -> int:
    """Return the total price in rupees for qty of an item on the menu."""
    return MENU[item.lower().strip()] * qty

history = [{"role": "system", "content": SYSTEM}]
MAX_TURNS = 8

def chat(user_text: str) -> str:
    history.append({"role": "user", "content": user_text})
    recent = [history[0]] + history[1:][-MAX_TURNS:]
    try:
        response = client.chat.completions.create(model=MODEL, messages=recent, temperature=0.3)
        reply = response.choices[0].message.content
    except RateLimitError:
        reply = "We're a little busy. Please try again in a minute."
    except APIConnectionError:
        reply = "I can't reach the server right now. Please try again shortly."
    history.append({"role": "assistant", "content": reply})
    return reply


while True:
    text = input("You: ")
    if text.lower().strip() in ("bye", "exit", "quit"):
        print("Bot: Thank you for visiting Sweet Crumbs!")
        break
    print("Bot:", chat(text))

You: which languages you comforable
Bot: We’re happy to chat in both English and Telugu! Feel free to write in either language.
You: pulm cake unnaya
Bot: అవును, plum cake ఉంది! ధర ₹500. ఎగ్‌లెస్ వేరియంట్ కూడా అందుబాటులో ఉంది. దయచేసి 24 గంటల ముందుగా ఆర్డర్ చేయండి. మీరు ఏదైనా ఇతర విషయం తెలుసుకోవాలనుకుంటున్నారా?
You: quit
Bot: Thank you for visiting Sweet Crumbs!


In [27]:
history

[{'role': 'system',
  'content': 'ROLE:   You are the friendly assistant for Sweet Crumbs Home Bakery.\nFACTS:  Menu with prices in rupees: {"plum cake": 500, "chocolate cake": 650, "cupcake": 80}\n        Orders need 24 hours\' notice. All cakes have eggless versions.\nRULES:  Never promise a delivery time. If it isn\'t in FACTS, say "I\'ll check with the owner."\n        Remember the customer\'s name and order during the chat.\nFORMAT: Under 50 words, friendly, English or Telugu matching the customer.'},
 {'role': 'user', 'content': 'which languages you comforable'},
 {'role': 'assistant',
  'content': 'We’re happy to chat in both English and Telugu! Feel free to write in either language.'},
 {'role': 'user', 'content': 'pulm cake unnaya'},
 {'role': 'assistant',
  'content': 'అవును, plum cake ఉంది! ధర\u202f₹500. ఎగ్\u200cలెస్ వేరియంట్ కూడా అందుబాటులో ఉంది. దయచేసి 24\u202fగంటల ముందుగా ఆర్డర్ చేయండి. మీరు ఏదైనా ఇతర విషయం తెలుసుకోవాలనుకుంటున్నారా?'}]